# Titanic - Predict Survival

**Goal:** Predict which passengers survived the Titanic disaster
**Algorithm:** Random Forest + Feature Engineering
**Dataset:** [Titanic Competition](https://www.kaggle.com/c/titanic)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
%matplotlib inline

In [1]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


Running locally - skipping Colab setup


## 1. Load Data from Kaggle

In [2]:
path = kagglehub.competition_download("titanic")
train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
print ('Train shape: %s' % (train.shape,))
print ('Test shape: %s' % (test.shape,))

Train shape: (891, 12)
Test shape: (418, 11)


<hr>## 2. Exploratory Data Analysis

In [3]:
print ('Survival distribution:\n%s' % train['Survived'].value_counts())
print ('\nSurvival rate: %.2f%%' % (train['Survived'].mean()*100))
print ('\nMissing values:')
print (train.isnull().sum()[train.isnull().sum() > 0])

Survival distribution:
0    549
1    342

Survival rate: 38.38%

Missing values:
Age        177
Cabin     687
Embarked    2


In [4]:
# Survival by gender and class
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
sns.barplot(x='Sex', y='Survived', data=train)
plt.title('Survival by Gender')

plt.subplot(1, 3, 2)
sns.barplot(x='Pclass', y='Survived', data=train)
plt.title('Survival by Class')

plt.subplot(1, 3, 3)
sns.histplot(x='Age', hue='Survived', data=train, kde=True, bins=30)
plt.title('Age Distribution by Survival')

plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 3. Feature Engineering

In [5]:
def engineer_features(df):
    data = df.copy()

    # Title from Name
    data['Title'] = data['Name'].apply(
        lambda x: re.search(r' ([A-Za-z]+)\.', x).group(1))
    title_map = {'Mr': 1, 'Mrs': 2, 'Miss': 3, 'Master': 4}
    data['Title'] = data['Title'].map(title_map).fillna(0).astype(int)

    # Family features
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)

    # Fill missing values
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    data['Embarked'] = data['Embarked'].fillna('S')

    # Encode categoricals
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    data['Embarked'] = data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    data['HasCabin'] = data['Cabin'].notna().astype(int)

    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']
    return data[features]

X_train_full = engineer_features(train)
y_train_full = train['Survived']
X_test_full = engineer_features(test)

print ('Features: %s' % list(X_train_full.columns))
print ('\nFirst 3 rows after engineering:\n%s' % X_train_full.head(3))

Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'HasCabin']


<hr>## 4. Train/Validation Split

In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42)
print ('Train: %d, Validation: %d' % (X_tr.shape[0], X_val.shape[0]))

Train: 712, Validation: 179


<hr>## 5. Train Model (with GridSearchCV)

In [7]:
params = {'n_estimators': [100, 200], 'max_depth': [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    params, cv=5, scoring='accuracy')
grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print ('Best parameters: %s' % grid.best_params_)
print ('Best CV score: %.4f' % grid.best_score_)

Best parameters: {'max_depth': 5, 'n_estimators': 200}
Best CV score: 0.8268


<hr>## 6. Evaluate Performance

In [8]:
y_pred = model.predict(X_val)
print ('Validation Accuracy: %.4f' % accuracy_score(y_val, y_pred))
print ('\nClassification Report:')
print (classification_report(y_val, y_pred, target_names=['Died', 'Survived']))

Validation Accuracy: 0.8324

Classification Report:
              precision    recall  f1-score   support

        Died       0.84      0.90      0.87       110
    Survived       0.81      0.72      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179


<hr>## 7. Feature Importance

In [9]:
importances = pd.DataFrame({
    'feature': X_train_full.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature', data=importances, palette='rocket')
plt.title('Feature Importance for Titanic Survival')
plt.tight_layout()
plt.show()
print (importances.to_string(index=False))

<Figure size NxN with 1 Axes>

     feature  importance     Sex       0.3245     Fare      0.1845     Age       0.1523     Title     0.1234     Pclass    0.0987     FamilySize 0.0543     HasCabin  0.0321


<hr>## 8. Predict Test Set & Save Submission

In [10]:
test_pred = model.predict(X_test_full)
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('titanic_submission.csv', index=False)

print ('Submission saved to titanic_submission.csv')
print ('Predicted survivors: %d / %d (%.1f%%)' % (
    test_pred.sum(), len(test_pred), test_pred.mean()*100))
print ('\nFirst 5 rows:\n%s' % submission.head())

Submission saved to titanic_submission.csv
Predicted survivors: 168 / 418 (40.2%)

First 5 rows:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         1
4          896         1
